# 04 — Transfer Entropy

Close the classical-information arc: make information flow **directed** — who drives whom — and meet the honesty caveats every estimate needs.

**Prerequisites:** `02/01_shannon_entropy`, `02/02_kl_divergence`, `02/03_mutual_information`.

**What you'll be able to do**
- Measure transfer entropy $\mathrm{TE}_{Y\to X} = I(X_{t+1}; Y_t \mid X_t)$ — directed information flow.
- Detect the true causal direction in a lagged-copy system.
- State why TE (and MI) estimates need surrogate checks, and why PID is not uniquely defined.

You have entropy (`02/01`), KL (`02/02`), and mutual information (`02/03`). MI tells you two signals are related, but not *which leads which*. Transfer entropy fixes that — and it is the measure most used to probe directed coupling in interacting systems, the kind of question hyperscanning asks of two brains.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.infotheory.classical import transfer_entropy

np.random.seed(42)
viz.use_course_style()

## 1. Directed flow: transfer entropy

Mutual information is symmetric — it cannot say who drives whom. **Transfer entropy**

$$ \mathrm{TE}_{Y\to X} = I\!\left(X_{t+1}; Y_t \mid X_t\right) $$

(Schreiber, 2000) is **directed**: it asks how much $Y$'s past tells us about $X$'s future, *beyond* what $X$'s own past already says. We build a source and a target that copies the source with a one-step lag, then measure flow in both directions.

In [ ]:
rng = np.random.default_rng(0)
source = rng.integers(0, 2, size=4000)
target = np.empty_like(source)
target[0] = 0
target[1:] = source[:-1]            # target copies source, lag 1

te_fwd = transfer_entropy(source, target)   # source -> target
te_bwd = transfer_entropy(target, source)   # target -> source
print(f"TE(source -> target) = {te_fwd:.3f} bits")
print(f"TE(target -> source) = {te_bwd:.3f} bits")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(["source -> target", "target -> source"], [te_fwd, te_bwd],
       color=[viz.FLOW_COLOR, viz.TARGET_COLOR], alpha=0.9)
ax.set_ylabel("transfer entropy (bits)")
ax.set_title("Transfer entropy detects the direction of influence", pad=12)
plt.show()

**Read the figure.** Although the two sequences are strongly related, transfer entropy is **large only in the true causal direction** (source → target $\approx 1$ bit) and $\approx 0$ backwards. Mutual information would have reported a single symmetric number; TE recovers the arrow of influence. This directionality is exactly why TE is used to probe information flow in coupled systems.

## 2. A word of caution

Two honesty notes, in keeping with this course's standard:

- **Estimation bias.** TE and mutual information estimated from finite samples are **biased** — short recordings systematically over- or under-estimate them. Always compare against shuffled / surrogate data to find the noise floor.
- **PID is not unique.** Partial information decomposition (Williams–Beer) promises to split information into *redundant*, *unique*, and *synergistic* parts — but there is **no agreed definition** of redundancy. Treat synergy/redundancy claims with care.

## 3. Dictionary update

The classical-information arc adds the mutual-information row; its quantum lift arrives in `02/10`.

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | Born rule $|\langle x|\psi\rangle|^2$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| Markov kernel | quantum channel (CPTP map) |
| independent variables | product state (no entanglement) |
| **mutual information $I(X;Y)$** | **quantum mutual information $I(A{:}B)$** |

## Your turn

1. **Noisy copy.** Make the target copy the source only 80% of the time (flip 20% of the lagged bits). Does $\mathrm{TE}_{\text{source}\to\text{target}}$ drop?
2. **Surrogate test.** Shuffle the source and recompute TE. Confirm it collapses to $\approx 0$ — that is the noise floor your real TE must beat to be meaningful.
3. **Bidirectional coupling.** Let the target depend on the source *and* the source depend slightly on the target's past. Do both TE directions become positive? Which is larger?

## What you built

- You measured transfer entropy and recovered the true direction of influence in a lagged-copy system.
- You learned the honesty checks: surrogate baselines for biased estimates, and caution around PID.
- You closed the classical-information arc and added the mutual-information row to the dictionary.

That completes the classical information toolbox — entropy, divergence, shared structure, directed flow. Beautifully done.

## What's next

So far every distance compared masses *bin by bin*, ignoring what the bins are. The geometry arc asks a deeper question: what does the *space* of all distributions look like? Next: `02/05_the_probability_simplex`.

## References

- T. Schreiber, "Measuring Information Transfer", *Physical Review Letters* 85, 461 (2000). DOI:10.1103/PhysRevLett.85.461
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006).
- P. L. Williams & R. D. Beer, "Nonnegative Decomposition of Multivariate Information", arXiv:1004.2515 (2010).

**Previous:** `notebooks/02_InformationAndGeometry/03_mutual_information.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/05_the_probability_simplex.ipynb`